In [1]:
import json
import re
from pathlib import Path

import pandas as pd

In [2]:
# Input and output directories
ROOT_DIR = Path("ctg_full_2005_2025")
RAW_DIR = ROOT_DIR / "raw_pages"
OUTPUT_DIR = ROOT_DIR / "cohort_qc"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Input file
RETRIEVAL_MANIFEST_FILE = ROOT_DIR / "download_manifest.json"


# Main cohort QC outputs
QC_SUMMARY_FILE = OUTPUT_DIR / "raw_cohort_qc_summary.json"

EXCLUSION_FILE = (
    OUTPUT_DIR / "eligibility_non_substantive_exclusions.csv"
)

EXCLUSION_SUMMARY_FILE = (
    OUTPUT_DIR / "eligibility_non_substantive_summary.csv"
)

EXCLUSION_REASON_FILE = (
    OUTPUT_DIR / "eligibility_non_substantive_reasons.csv"
)

FINAL_COHORT_FILE = OUTPUT_DIR / "analytic_cohort.csv"

In [3]:
EXPECTED_START_YEAR_MIN = 2005
EXPECTED_START_YEAR_MAX = 2025
EXPECTED_STUDY_TYPE = "INTERVENTIONAL"

EXPECTED_QUERY = (
    "AREA[StudyType]INTERVENTIONAL "
    "AND AREA[StartDate]RANGE[2005-01-01, 2025-12-31]"
)

EXPECTED_START_DATE = "2005-01-01"
EXPECTED_END_DATE = "2025-12-31"

REQUIRED_RETRIEVAL_FIELDS = {
    "NCTId",
    "StudyType",
    "OverallStatus",
    "StartDate",
    "StudyFirstPostDate",
    "Phase",
    "Condition",
    "ConditionMesh",
    "ConditionAncestor",
    "LeadSponsorName",
    "LeadSponsorClass",
    "EnrollmentCount",
    "Sex",
    "MinimumAge",
    "MaximumAge",
    "LocationCountry",
    "LocationFacility",
    "LocationCity",
    "LocationState",
    "LocationZip",
    "EligibilityCriteria",
}

In [4]:
def get_nested(dictionary, *keys, default=None):
    """Safely retrieve a value from a nested dictionary."""

    current = dictionary

    for key in keys:
        if not isinstance(current, dict) or key not in current:
            return default

        current = current[key]

    return current


def classify_date_precision(date_value):
    """
    Classify a ClinicalTrials.gov date as DAY, MONTH, YEAR,
    MISSING, or OTHER.
    """

    if not isinstance(date_value, str) or not date_value.strip():
        return "MISSING"

    value = date_value.strip()

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
        return "DAY"

    if re.fullmatch(r"\d{4}-\d{2}", value):
        return "MONTH"

    if re.fullmatch(r"\d{4}", value):
        return "YEAR"

    return "OTHER"


def extract_year(date_value):
    """Extract the four-digit year from a ClinicalTrials.gov date."""

    if not isinstance(date_value, str):
        return None

    match = re.match(r"^(\d{4})", date_value.strip())

    if not match:
        return None

    return int(match.group(1))


def has_nonempty_text(value):
    """Return whether a value contains non-whitespace text."""

    return isinstance(value, str) and bool(value.strip())

In [5]:
# Load the retrieval manifest
if not RETRIEVAL_MANIFEST_FILE.exists():
    raise RuntimeError(
        f"Retrieval manifest not found: {RETRIEVAL_MANIFEST_FILE}"
    )

with open(RETRIEVAL_MANIFEST_FILE, "r", encoding="utf-8") as f:
    retrieval_manifest = json.load(f)

if retrieval_manifest.get("query") != EXPECTED_QUERY:
    raise RuntimeError(
        "Retrieval query does not match the predefined cohort query.\n"
        f"Expected: {EXPECTED_QUERY}\n"
        f"Found: {retrieval_manifest.get('query')}"
    )

if retrieval_manifest.get("start_date") != EXPECTED_START_DATE:
    raise RuntimeError(
        "Retrieval start date does not match the predefined cohort.\n"
        f"Expected: {EXPECTED_START_DATE}\n"
        f"Found: {retrieval_manifest.get('start_date')}"
    )    
    
if retrieval_manifest.get("end_date") != EXPECTED_END_DATE:
    raise RuntimeError(
        "Retrieval end date does not match the predefined cohort.\n"
        f"Expected: {EXPECTED_END_DATE}\n"
        f"Found: {retrieval_manifest.get('end_date')}"
    )   

retrieved_fields = set(retrieval_manifest.get("fields", []))

missing_fields = REQUIRED_RETRIEVAL_FIELDS - retrieved_fields

if missing_fields:
    raise RuntimeError(
        "Required fields were not included in the retrieval: "
        + ", ".join(sorted(missing_fields))
    )    
    
if not retrieval_manifest.get("download_complete"):
    raise RuntimeError(
        "The retrieval manifest does not indicate a complete download."
    )


# Locate raw API page files
page_files = sorted(RAW_DIR.glob("page_*.json"))

if not page_files:
    raise RuntimeError(
        f"No page_*.json files found in {RAW_DIR.resolve()}"
    )


expected_pages = retrieval_manifest.get("completed_pages")

if expected_pages is None:
    raise RuntimeError(
        "The retrieval manifest does not contain completed_pages."
    )

if len(page_files) != expected_pages:
    raise RuntimeError(
        "Raw page count does not match the retrieval manifest: "
        f"{len(page_files)} files found versus "
        f"{expected_pages} expected."
    )


print(f"Raw page files found: {len(page_files)}")
print(
    f"Retrieval manifest records: "
    f"{retrieval_manifest.get('records_on_disk')}"
)

Raw page files found: 406
Retrieval manifest records: 405911


In [6]:
rows = []
page_study_counts = []

for page_index, page_file in enumerate(page_files, start=1):

    with open(page_file, "r", encoding="utf-8") as f:
        page_data = json.load(f)

    if page_data.get("query") != EXPECTED_QUERY:
        raise RuntimeError(
            f"Retrieval query mismatch in raw page: {page_file.name}"
        )    
        
    if page_data.get("page_number") != page_index:
        raise RuntimeError(
            f"Page-number mismatch in raw page: {page_file.name}. "
            f"Expected {page_index}, found {page_data.get('page_number')}."
        )
    
    if "studies" not in page_data:
        raise RuntimeError(
            f"'studies' field missing from raw page: {page_file.name}"
        )

    studies = page_data["studies"]
    
    if page_data.get("n_studies") != len(studies):
        raise RuntimeError(
            f"Study-count mismatch in raw page: {page_file.name}. "
            f"Stored {page_data.get('n_studies')}, found {len(studies)}."
        )

    if not isinstance(studies, list):
        raise RuntimeError(
            f"'studies' is not a list in raw page: {page_file.name}"
        )

    if not studies:
        raise RuntimeError(
            f"Raw page contains no studies: {page_file.name}"
        )

    page_study_counts.append(len(studies))

    for study in studies:

        protocol = study.get("protocolSection", {})

        identification = protocol.get("identificationModule", {})
        status = protocol.get("statusModule", {})
        design = protocol.get("designModule", {})
        conditions_module = protocol.get("conditionsModule", {})
        sponsor_module = protocol.get("sponsorCollaboratorsModule", {})
        eligibility_module = protocol.get("eligibilityModule", {})
        locations_module = protocol.get("contactsLocationsModule", {})

        derived = study.get("derivedSection", {})

        condition_browse_module = derived.get(
            "conditionBrowseModule",
            {},
        )
        
        # Core identifier
        nct_id = identification.get("nctId")

        # Dates
        start_date = get_nested(
            status,
            "startDateStruct",
            "date",
        )

        first_posted_date = get_nested(
            status,
            "studyFirstPostDateStruct",
            "date",
        )

        start_year = extract_year(start_date)
        first_posted_year = extract_year(first_posted_date)

        start_date_precision = classify_date_precision(start_date)

        # Study characteristics
        phases = design.get("phases", []) or []
        conditions = conditions_module.get("conditions", []) or []
        condition_meshes = (
            condition_browse_module.get("meshes", []) or []
        )
        condition_ancestors = (
            condition_browse_module.get("ancestors", []) or []
        )
        lead_sponsor = sponsor_module.get("leadSponsor", {}) or {}

        # Locations and countries
        locations = locations_module.get("locations", []) or []

        countries = sorted(
            {
                location.get("country")
                for location in locations
                if (
                    isinstance(location, dict)
                    and location.get("country")
                )
            }
        )

        # Eligibility text
        eligibility_text = eligibility_module.get(
            "eligibilityCriteria"
        )

        eligibility_available = has_nonempty_text(
            eligibility_text
        )

        eligibility_char_count = (
            len(eligibility_text.strip())
            if eligibility_available
            else None
        )

        # Expected-cohort checks
        study_type = design.get("studyType")

        study_type_expected = (
            study_type == EXPECTED_STUDY_TYPE
        )

        start_year_in_range = (
            isinstance(start_year, int)
            and EXPECTED_START_YEAR_MIN
            <= start_year
            <= EXPECTED_START_YEAR_MAX
        )

        # Study-level inventory row
        rows.append(
            {
                "nct_id": nct_id,
                "source_page": page_file.name,
                "study_type": study_type,
                "study_type_expected": study_type_expected,
                "study_status": status.get("overallStatus"),
                "study_start_date": start_date,
                "study_start_year": start_year,
                "start_date_precision": start_date_precision,
                "start_year_in_range": start_year_in_range,
                "first_posted_date": first_posted_date,
                "first_posted_year": first_posted_year,
                "phase": "|".join(phases) if phases else None,
                "conditions": json.dumps(
                    conditions,
                    ensure_ascii=False,
                ),
                "condition_meshes": json.dumps(
                    condition_meshes,
                    ensure_ascii=False,
                ),
                "condition_ancestors": json.dumps(
                    condition_ancestors,
                    ensure_ascii=False,
                ),
                "lead_sponsor": lead_sponsor.get("name"),
                "sponsor_class": lead_sponsor.get("class"),
                "enrollment": get_nested(
                    design,
                    "enrollmentInfo",
                    "count",
                ),
                "sex": eligibility_module.get("sex"),
                "minimum_age": eligibility_module.get("minimumAge"),
                "maximum_age": eligibility_module.get("maximumAge"),
                "countries": json.dumps(
                    countries,
                    ensure_ascii=False,
                ),
                "n_countries": len(countries),
                "locations": json.dumps(
                    locations,
                    ensure_ascii=False,
                ),
                "eligibility_available": eligibility_available,
                "eligibility_char_count": eligibility_char_count,
                "eligibility_text": eligibility_text,
            }
        )

    if page_index % 25 == 0 or page_index == len(page_files):
        print(
            f"Processed pages: "
            f"{page_index}/{len(page_files)}"
        )

Processed pages: 25/406
Processed pages: 50/406
Processed pages: 75/406
Processed pages: 100/406
Processed pages: 125/406
Processed pages: 150/406
Processed pages: 175/406
Processed pages: 200/406
Processed pages: 225/406
Processed pages: 250/406
Processed pages: 275/406
Processed pages: 300/406
Processed pages: 325/406
Processed pages: 350/406
Processed pages: 375/406
Processed pages: 400/406
Processed pages: 406/406


In [7]:
inventory_df = pd.DataFrame(rows)

print(
    f"Study records loaded into inventory: "
    f"{len(inventory_df)}"
)

Study records loaded into inventory: 405911


In [8]:
# Basic reconciliation
n_records = len(inventory_df)

n_unique_ids = inventory_df["nct_id"].nunique(dropna=True)

n_missing_ids = int(
    inventory_df["nct_id"].isna().sum()
)

n_duplicate_ids = int(
    inventory_df["nct_id"]
    .duplicated(keep=False)
    .sum()
)


# Eligibility availability
n_eligibility_available = int(
    inventory_df["eligibility_available"].sum()
)

n_eligibility_unavailable = (
    n_records - n_eligibility_available
)


# Cohort-definition validation
n_other_date_precision = int(
    (inventory_df["start_date_precision"] == "OTHER").sum()
)

if n_other_date_precision != 0:
    raise RuntimeError(
        "Unexpected Study Start Date format found: "
        f"{n_other_date_precision} records."
    )
    
n_wrong_study_type = int(
    (~inventory_df["study_type_expected"]).sum()
)

n_start_year_out_of_range = int(
    (~inventory_df["start_year_in_range"]).sum()
)


# Reconcile against the completed retrieval
expected_records = retrieval_manifest.get("records_on_disk")
expected_unique_ids = retrieval_manifest.get("unique_nct_ids")

if expected_records is None or expected_unique_ids is None:
    raise RuntimeError(
        "Retrieval manifest is missing record-count reconciliation fields."
    )

if n_records != expected_records:
    raise RuntimeError(
        "Study-count mismatch between raw cohort QC and retrieval manifest: "
        f"{n_records} records loaded versus {expected_records} expected."
    )

if n_missing_ids != 0:
    raise RuntimeError(
        f"Missing NCT IDs found in the raw cohort: {n_missing_ids}"
    )

if n_duplicate_ids != 0:
    raise RuntimeError(
        f"Duplicate NCT-ID rows found in the raw cohort: {n_duplicate_ids}"
    )

if n_unique_ids != expected_unique_ids:
    raise RuntimeError(
        "Unique NCT-ID count does not match the retrieval manifest: "
        f"{n_unique_ids} found versus {expected_unique_ids} expected."
    )

if n_unique_ids != n_records:
    raise RuntimeError(
        "The number of unique NCT IDs does not equal the number of study records."
    )

if n_wrong_study_type != 0:
    raise RuntimeError(
        f"Studies outside the expected study type were found: "
        f"{n_wrong_study_type}"
    )

if n_start_year_out_of_range != 0:
    raise RuntimeError(
        "Studies with missing or out-of-range Study Start Year were found: "
        f"{n_start_year_out_of_range}"
    )


print("Raw cohort integrity checks passed.")
print(f"Records: {n_records}")
print(f"Unique NCT IDs: {n_unique_ids}")
print(f"Eligibility text available: {n_eligibility_available}")
print(f"Eligibility text unavailable: {n_eligibility_unavailable}")

Raw cohort integrity checks passed.
Records: 405911
Unique NCT IDs: 405911
Eligibility text available: 405898
Eligibility text unavailable: 13


In [9]:
# Study counts by Study Start Year
year_counts = (
    inventory_df["study_start_year"]
    .value_counts(dropna=False)
    .rename_axis("study_start_year")
    .reset_index(name="n_studies")
    .sort_values("study_start_year", na_position="last")
)

year_counts.to_csv(
    OUTPUT_DIR / "study_counts_by_start_year.csv",
    index=False,
    encoding="utf-8-sig",
)


# Study Start Date precision
date_precision_df = (
    inventory_df["start_date_precision"]
    .value_counts(dropna=False)
    .rename_axis("start_date_precision")
    .reset_index(name="n_studies")
)

date_precision_df.to_csv(
    OUTPUT_DIR / "start_date_precision_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

In [10]:
eligibility_by_year = (
    inventory_df
    .groupby("study_start_year", dropna=False)
    .agg(
        n_studies=("nct_id", "size"),
        n_eligibility_available=("eligibility_available", "sum"),
    )
    .reset_index()
)

eligibility_by_year["n_eligibility_unavailable"] = (
    eligibility_by_year["n_studies"]
    - eligibility_by_year["n_eligibility_available"]
)

eligibility_by_year["eligibility_available_percent"] = (
    eligibility_by_year["n_eligibility_available"]
    / eligibility_by_year["n_studies"]
    * 100
)

eligibility_by_year.to_csv(
    OUTPUT_DIR / "eligibility_availability_by_year.csv",
    index=False,
    encoding="utf-8-sig",
)

In [11]:
metadata_columns = [
    "study_start_date",
    "first_posted_date",
    "study_status",
    "phase",
    "conditions",
    "lead_sponsor",
    "sponsor_class",
    "enrollment",
    "sex",
    "minimum_age",
    "maximum_age",
    "countries",
]

missingness_rows = []

for column in metadata_columns:
    series = inventory_df[column]

    if column in {"conditions", "countries"}:
        missing_mask = (
            series.isna()
            | series.isin(["[]", ""])
        )
    else:
        missing_mask = (
            series.isna()
            | (series.astype(str).str.strip() == "")
        )

    n_missing = int(missing_mask.sum())

    missingness_rows.append(
        {
            "variable": column,
            "n_missing": n_missing,
            "percent_missing": n_missing / n_records * 100,
        }
    )

missingness_df = pd.DataFrame(missingness_rows)

missingness_df.to_csv(
    OUTPUT_DIR / "metadata_missingness.csv",
    index=False,
    encoding="utf-8-sig",
)

In [12]:
qc_summary = {
    "n_records": n_records,
    "n_unique_nct_ids": n_unique_ids,
    "n_missing_nct_ids": n_missing_ids,
    "n_duplicate_nct_id_rows": n_duplicate_ids,
    "n_wrong_study_type": n_wrong_study_type,
    "n_start_year_out_of_range_or_missing": n_start_year_out_of_range,
    "n_eligibility_available": n_eligibility_available,
    "n_eligibility_unavailable": n_eligibility_unavailable,
    "start_date_precision": {
        str(row["start_date_precision"]): int(row["n_studies"])
        for _, row in date_precision_df.iterrows()
    },
}

with open(QC_SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(
        qc_summary,
        f,
        ensure_ascii=False,
        indent=2,
    )


print()
print("=" * 70)
print("RAW COHORT QC SUMMARY")
print("=" * 70)
print(f"Records: {n_records}")
print(f"Unique NCT IDs: {n_unique_ids}")
print(f"Missing NCT IDs: {n_missing_ids}")
print(f"Duplicate NCT-ID rows: {n_duplicate_ids}")
print(f"Wrong study type: {n_wrong_study_type}")
print(
    "Start year out of range or missing: "
    f"{n_start_year_out_of_range}"
)
print(
    f"Eligibility text unavailable: "
    f"{n_eligibility_unavailable}"
)
print()
print(f"QC summary saved to: {QC_SUMMARY_FILE}")


RAW COHORT QC SUMMARY
Records: 405911
Unique NCT IDs: 405911
Missing NCT IDs: 0
Duplicate NCT-ID rows: 0
Wrong study type: 0
Start year out of range or missing: 0
Eligibility text unavailable: 13

QC summary saved to: ctg_full_2005_2025/cohort_qc/raw_cohort_qc_summary.json


In [13]:
def normalize_for_non_substantive_qc(text):
    """
    Normalize eligibility text for whole-field non-substantive QC only.

    This function is not part of the analytical preprocessing pipeline.
    """

    if not has_nonempty_text(text):
        return ""

    text = (
        text.replace("\r\n", "\n")
        .replace("\r", "\n")
        .replace("\t", " ")
    )

    replacements = {
        r"\>": ">",
        r"\<": "<",
        r"\[": "[",
        r"\]": "]",
        r"\&": "&",
        r"\-": "-",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    # Remove standalone inclusion/exclusion headings.
    text = re.sub(
        r"""
        (?im)
        ^[ \t]*
        (?:inclusion|exclusion)
        [ \t]+criteria
        [ \t]*:?
        [ \t]*$
        """,
        "",
        text,
        flags=re.VERBOSE,
    )

    # Remove structural list markers at line starts.
    text = re.sub(
        r"""
        (?m)
        ^[ \t]*
        (?:
            [*•-]
            |
            \d{1,3}[.)]
            |
            [A-Za-z][.)]
            |
            [ivxlcdmIVXLCDM]{1,6}[.)]
        )
        [ \t]*
        """,
        "",
        text,
        flags=re.VERBOSE,
    )

    # Remove inline inclusion/exclusion heading labels.
    text = re.sub(
        r"""
        (?i)
        \b
        (?:inclusion|exclusion)
        [ \t]+criteria
        [ \t]*:
        """,
        " ",
        text,
        flags=re.VERBOSE,
    )

    text = re.sub(r"\s+", " ", text)

    return text.strip()


def simplify_for_fullmatch(text):
    """
    Create a simplified whole-field comparison form while retaining words.
    """

    text = text.lower().strip()
    text = text.replace("’", "'")

    text = re.sub(r"[;,|]+", " ", text)

    # Preserve "/" so placeholders such as "N/A" remain recognizable.
    text = re.sub(
        r"^[\s.:!?()\[\]{}\-]+",
        "",
        text,
    )

    text = re.sub(
        r"[\s.:!?()\[\]{}\-]+$",
        "",
        text,
    )

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [14]:
# Recognized whole-field placeholders
PLACEHOLDER_ATOM = (
    r"(?:"
    r"n/?a"
    r"|na"
    r"|none"
    r"|nil"
    r"|not\s+applicable"
    r"|not\s+available"
    r"|unavailable"
    r"|unknown"
    r"|0"
    r")"
)


PLACEHOLDER_ONLY_RE = re.compile(
    rf"^{PLACEHOLDER_ATOM}$",
    flags=re.IGNORECASE,
)


PLACEHOLDERS_ONLY_RE = re.compile(
    rf"""
    ^
    {PLACEHOLDER_ATOM}
    (?:
        \s*
        (?:[,;/|.]+|\s+)
        \s*
        {PLACEHOLDER_ATOM}
    )*
    $
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

In [15]:
REDIRECT_ONLY_PATTERNS = [
    (
        "CONTACT_SPONSOR",
        re.compile(
            r"""
            ^
            (?:please\s+)?
            contact
            \s+
            (?:the\s+)?
            sponsor
            $
            """,
            flags=re.IGNORECASE | re.VERBOSE,
        ),
    ),
    (
        "SEE_PROTOCOL",
        re.compile(
            r"""
            ^
            (?:please\s+)?
            see
            \s+
            (?:the\s+)?
            protocol
            $
            """,
            flags=re.IGNORECASE | re.VERBOSE,
        ),
    ),
    (
        "REFER_TO_PROTOCOL",
        re.compile(
            r"""
            ^
            (?:please\s+)?
            refer
            \s+to
            \s+
            (?:the\s+)?
            protocol
            $
            """,
            flags=re.IGNORECASE | re.VERBOSE,
        ),
    ),
    (
        "CRITERIA_DEFINED_IN_PROTOCOL",
        re.compile(
            r"""
            ^
            (?:eligibility\s+)?
            criteria
            \s+
            (?:are\s+)?
            (?:defined|described|specified)
            \s+
            (?:in|within)
            \s+
            (?:the\s+)?
            (?:study\s+|implementation\s+)?
            protocol
            $
            """,
            flags=re.IGNORECASE | re.VERBOSE,
        ),
    ),
    (
        "AS_PER_PROTOCOL",
        re.compile(
            r"""
            ^
            as
            \s+
            (?:per|defined\s+in|specified\s+in)
            \s+
            (?:the\s+)?
            protocol
            $
            """,
            flags=re.IGNORECASE | re.VERBOSE,
        ),
    ),
]

In [16]:
ADMIN_ONLY_PATTERNS = [
    (
        "CLOSED_TO_ACCRUAL",
        re.compile(
            r"""
            ^
            (?:study\s+|trial\s+)?
            closed
            \s+to
            \s+accrual
            $
            """,
            flags=re.IGNORECASE | re.VERBOSE,
        ),
    ),
    (
        "NO_PATIENTS_ENROLLED",
        re.compile(
            r"""
            ^
            no
            \s+
            (?:patients?|participants?|subjects?)
            \s+
            (?:were\s+|have\s+been\s+)?
            enrolled
            $
            """,
            flags=re.IGNORECASE | re.VERBOSE,
        ),
    ),
    (
        "NO_ENROLLMENT",
        re.compile(
            r"""
            ^
            no
            \s+
            enrollment
            $
            """,
            flags=re.IGNORECASE | re.VERBOSE,
        ),
    ),
    (
        "NOT_RECRUITING",
        re.compile(
            r"""
            ^
            (?:study\s+|trial\s+)?
            not
            \s+
            recruiting
            $
            """,
            flags=re.IGNORECASE | re.VERBOSE,
        ),
    ),
]

In [17]:
def classify_non_substantive_eligibility(text):
    """
    Classify the entire eligibility-criteria field for cohort QC.

    A record is flagged only when the whole field is missing, empty,
    structural-only, placeholder-only, redirect-only, or
    administrative/status-only.
    """

    result = {
        "qc_flagged": False,
        "qc_category": None,
        "qc_reason": None,
        "qc_normalized_text": None,
        "qc_simplified_text": None,
    }

    # 1. Missing or empty
    if not has_nonempty_text(text):
        result.update(
            {
                "qc_flagged": True,
                "qc_category": "MISSING_OR_EMPTY",
                "qc_reason": "Eligibility field missing or empty",
                "qc_normalized_text": "",
                "qc_simplified_text": "",
            }
        )
        return result

    normalized = normalize_for_non_substantive_qc(text)
    simplified = simplify_for_fullmatch(normalized)

    result["qc_normalized_text"] = normalized
    result["qc_simplified_text"] = simplified

    # 2. Headings or structural markers only
    if not simplified:
        result.update(
            {
                "qc_flagged": True,
                "qc_category": "HEADINGS_ONLY",
                "qc_reason": (
                    "No substantive text remains after structural "
                    "headings/list markers are removed"
                ),
            }
        )
        return result

    # 3. Single recognized placeholder only
    if PLACEHOLDER_ONLY_RE.fullmatch(simplified):
        result.update(
            {
                "qc_flagged": True,
                "qc_category": "PLACEHOLDER_ONLY",
                "qc_reason": (
                    "Eligibility field consists only of "
                    "a recognized placeholder"
                ),
            }
        )
        return result

    # 4. One or more recognized placeholders only
    if PLACEHOLDERS_ONLY_RE.fullmatch(simplified):
        result.update(
            {
                "qc_flagged": True,
                "qc_category": "PLACEHOLDERS_ONLY",
                "qc_reason": (
                    "Eligibility field consists only of "
                    "recognized placeholders"
                ),
            }
        )
        return result

    # 5. Redirect-only text
    for reason, pattern in REDIRECT_ONLY_PATTERNS:
        if pattern.fullmatch(simplified):
            result.update(
                {
                    "qc_flagged": True,
                    "qc_category": "REDIRECT_ONLY",
                    "qc_reason": reason,
                }
            )
            return result

    # 6. Administrative or recruitment-status-only text
    for reason, pattern in ADMIN_ONLY_PATTERNS:
        if pattern.fullmatch(simplified):
            result.update(
                {
                    "qc_flagged": True,
                    "qc_category": "ADMINISTRATIVE_ONLY",
                    "qc_reason": reason,
                }
            )
            return result

    return result

In [18]:
classification_results = inventory_df["eligibility_text"].apply(
    classify_non_substantive_eligibility
)

classification_df = pd.DataFrame(
    classification_results.tolist(),
    index=inventory_df.index,
)

cohort_qc_df = pd.concat(
    [
        inventory_df,
        classification_df,
    ],
    axis=1,
)

In [19]:
flagged_df = cohort_qc_df.loc[
    cohort_qc_df["qc_flagged"]
].copy()

In [20]:
flagged_columns = [
    "nct_id",
    "source_page",
    "study_start_year",
    "qc_category",
    "qc_reason",
    "eligibility_char_count",
    "qc_normalized_text",
    "qc_simplified_text",
    "eligibility_text",
]

flagged_df[flagged_columns].to_csv(
    EXCLUSION_FILE,
    index=False,
    encoding="utf-8-sig",
)

In [21]:
category_summary = (
    flagged_df["qc_category"]
    .value_counts()
    .rename_axis("qc_category")
    .reset_index(name="n_records")
)

category_summary.to_csv(
    EXCLUSION_SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig",
)


reason_summary = (
    flagged_df
    .groupby(
        ["qc_category", "qc_reason"],
        dropna=False,
    )
    .size()
    .reset_index(name="n_records")
    .sort_values(
        ["qc_category", "n_records"],
        ascending=[True, False],
    )
)

reason_summary.to_csv(
    EXCLUSION_REASON_FILE,
    index=False,
    encoding="utf-8-sig",
)


print()
print("=" * 70)
print("NON-SUBSTANTIVE ELIGIBILITY QC")
print("=" * 70)
print(f"Total flagged records: {len(flagged_df)}")
print()
print(category_summary.to_string(index=False))
print()
print("Reason summary:")
print(reason_summary.to_string(index=False))


NON-SUBSTANTIVE ELIGIBILITY QC
Total flagged records: 36

        qc_category  n_records
      HEADINGS_ONLY         13
   MISSING_OR_EMPTY         13
  PLACEHOLDERS_ONLY          6
ADMINISTRATIVE_ONLY          2
      REDIRECT_ONLY          2

Reason summary:
        qc_category                                                                      qc_reason  n_records
ADMINISTRATIVE_ONLY                                                              CLOSED_TO_ACCRUAL          1
ADMINISTRATIVE_ONLY                                                           NO_PATIENTS_ENROLLED          1
      HEADINGS_ONLY No substantive text remains after structural headings/list markers are removed         13
   MISSING_OR_EMPTY                                             Eligibility field missing or empty         13
  PLACEHOLDERS_ONLY                     Eligibility field consists only of recognized placeholders          6
      REDIRECT_ONLY                                                           

In [22]:
# Construct the final analytic cohort
analytic_cohort_df = cohort_qc_df.loc[
    ~cohort_qc_df["qc_flagged"]
].copy()

n_excluded = len(flagged_df)
n_analytic = len(analytic_cohort_df)


# Final cohort reconciliation
if n_records != n_excluded + n_analytic:
    raise RuntimeError(
        "Final cohort reconciliation failed: "
        f"{n_records} raw studies != "
        f"{n_excluded} excluded + {n_analytic} retained."
    )

if analytic_cohort_df["nct_id"].isna().any():
    raise RuntimeError(
        "Missing NCT IDs found in the final analytic cohort."
    )

if analytic_cohort_df["nct_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate NCT IDs found in the final analytic cohort."
    )

if analytic_cohort_df["qc_flagged"].any():
    raise RuntimeError(
        "A flagged eligibility record remains in the final analytic cohort."
    )


print()
print("=" * 70)
print("FINAL ANALYTIC COHORT")
print("=" * 70)
print(f"Raw cohort: {n_records}")
print(f"Excluded: {n_excluded}")
print(f"Final analytic cohort: {n_analytic}")
print()
print("Final cohort reconciliation passed.")


FINAL ANALYTIC COHORT
Raw cohort: 405911
Excluded: 36
Final analytic cohort: 405875

Final cohort reconciliation passed.


In [23]:
final_columns = list(inventory_df.columns)

analytic_cohort_output_df = analytic_cohort_df[
    final_columns
].copy()

analytic_cohort_output_df.to_csv(
    FINAL_COHORT_FILE,
    index=False,
    encoding="utf-8-sig",
)

print(f"Final analytic cohort saved to: {FINAL_COHORT_FILE}")

Final analytic cohort saved to: ctg_full_2005_2025/cohort_qc/analytic_cohort.csv


In [24]:
qc_summary["eligibility_exclusions"] = {
    "n_excluded": n_excluded,
    "category_counts": {
        str(row["qc_category"]): int(row["n_records"])
        for _, row in category_summary.iterrows()
    },
}

qc_summary["final_analytic_cohort"] = {
    "n_studies": n_analytic,
    "n_unique_nct_ids": int(
        analytic_cohort_output_df["nct_id"].nunique()
    ),
}

with open(QC_SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(
        qc_summary,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Final QC summary saved to: {QC_SUMMARY_FILE}")

Final QC summary saved to: ctg_full_2005_2025/cohort_qc/raw_cohort_qc_summary.json
